In [20]:
from utils.io import read_all_df
import numpy.ma as ma
import pandas as pd
import os

In [18]:
# 多個 category
categories = [
    "Selected","SelectedVer2", 'Top50','Top100','BioMedTSE','ChemicalTSE','ElectronicComponentTSE',
    'FinanceTSE','OptoTSE',"SemiconductorTSE", "BuildingTSE","CarTSE","ClothesTSE","NetworkTSE","MetalTSE",
    "EETSE","ComputerTSE"
]
data = ["Dataset_Pct", "Dataset_Abs"]
loss = ["mse", "ccc", "rank"]

len(categories) * len(data) * len(loss)

102

In [ ]:
RESULT_ROOT_DIR = "results"

In [11]:
from analysis.metrics.correlations import daily_correlation, stockly_correlation, total_correlation

def analyze(pred_df, truth_df):
    return {
        "daily_correlation": daily_correlation(pred_df, truth_df),
        "stockly_correlation": stockly_correlation(pred_df, truth_df),
        "total_correlation": total_correlation(pred_df, truth_df)
    }

In [ ]:
from itertools import product
datas = []
for category, d, l in product(categories, data, loss):
    path = os.path.join(RESULT_ROOT_DIR, f"{category}_{d}_{l}")
    pred_df, truth_df, pred_val, truth_val = read_all_df(path)
    if pred_df is not None:
        row_data = {
            "category": category,
            "data": d,
            "loss": l
        }
        row_data.update(analyze(pred_df, truth_df))
        datas.append(row_data)
    else:
        print(f"No data found for Category: {category}, Data: {d}, Loss: {l}\n")

In [19]:
result_df = pd.DataFrame(datas)
result_df.groupby(["data", "loss"]).agg(
    daily_correlation_mean=('daily_correlation', 'mean'),
    stockly_correlation_mean=('stockly_correlation', 'mean'),
    total_correlation_mean=('total_correlation', 'mean')
)

daily_correlation_mean  stockly_correlation_mean  \
data        loss                                                     
Dataset_Abs ccc                 0.185458                  0.089903   
            mse                 0.210204                  0.070779   
            rank               -0.080547                  0.023857   
Dataset_Pct ccc                 0.050149                  0.023162   
            mse                 0.002962                 -0.004982   
            rank                0.085886                  0.036255   

                  total_correlation_mean  
data        loss                          
Dataset_Abs ccc                 0.180177  
            mse                 0.181741  
            rank               -0.039713  
Dataset_Pct ccc                 0.024695  
            mse                -0.003961  
            rank                0.044956